# Cash Flow Revenue Forecast - 2018

Daily cash flow revenue forecast for 8 customers (Company A through H) with specific payment terms.

In [ ]:
import pandas as pd
import numpy as np
from datetime import date, timedelta
import calendar

In [ ]:
# Create date range covering 2017-2019 for full modeling
date_range = pd.date_range(start='2017-01-01', end='2019-12-31', freq='D')
df = pd.DataFrame({'date': date_range})
df['weekday'] = df['date'].dt.weekday  # 0=Mon, 6=Sun
df['is_weekday'] = df['weekday'] < 5
df['day'] = df['date'].dt.day
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year
df['day_name'] = df['date'].dt.day_name()
df.set_index('date', inplace=True)
print(f"Date range: {df.index[0]} to {df.index[-1]}, total {len(df)} days")

In [ ]:
# ============================================================
# COMPANY A
# Monthly figure of $17.3m paid in 4 segments:
# 11% on day 5, 21% on day 10, 41% on day 18, balance (27%) on day 23
# Payments made even on weekends
# ============================================================
df['company_a'] = 0.0
monthly_a = 17.3  # $m

for idx in df.index:
    d = idx.day
    if d == 5:
        df.loc[idx, 'company_a'] = monthly_a * 0.11
    elif d == 10:
        df.loc[idx, 'company_a'] = monthly_a * 0.21
    elif d == 18:
        df.loc[idx, 'company_a'] = monthly_a * 0.41
    elif d == 23:
        df.loc[idx, 'company_a'] = monthly_a * 0.27

print(f"Company A total 2018: ${df.loc['2018', 'company_a'].sum():.2f}m")

In [ ]:
# ============================================================
# COMPANY B
# Daily charge of $0.35m accrues daily.
# Paid monthly on the 3rd Thursday of the month.
# Accrual period: from day after 3rd Thursday of prior month
# to 3rd Thursday of current month (inclusive).
# The spec example says "from the 3rd Friday" but that is only
# correct when the 3rd Friday immediately follows the 3rd Thursday
# in the prior month. The general rule is: the accrual starts the
# day after the previous payment (3rd Thu of prior month + 1 day).
# ============================================================
df['company_b'] = 0.0
daily_b = 0.35  # $m

def nth_weekday_of_month(year, month, weekday, n):
    """Find the nth occurrence of a weekday in a given month.
    weekday: 0=Mon, 1=Tue, ..., 3=Thu, 4=Fri, 5=Sat, 6=Sun
    n: 1-based (1=first, 2=second, 3=third)
    """
    first_day = date(year, month, 1)
    first_weekday = first_day.weekday()
    days_until = (weekday - first_weekday) % 7
    first_occurrence = first_day + timedelta(days=days_until)
    target = first_occurrence + timedelta(weeks=n-1)
    return target

# For each month, find 3rd Thursday (payment date) and compute accrual.
# Accrual: from (3rd Thursday of prior month + 1 day) to 3rd Thursday
# of current month (inclusive).
for year in range(2017, 2020):
    for month in range(1, 13):
        third_thu = nth_weekday_of_month(year, month, 3, 3)  # 3=Thursday
        # Prior month
        if month == 1:
            prev_year, prev_month = year - 1, 12
        else:
            prev_year, prev_month = year, month - 1
        third_thu_prev = nth_weekday_of_month(prev_year, prev_month, 3, 3)  # 3=Thursday
        
        # Accrual start = day after 3rd Thursday of prior month
        accrual_start = third_thu_prev + timedelta(days=1)
        
        # Number of days from accrual_start to 3rd Thursday of current month (inclusive)
        num_days = (third_thu - accrual_start).days + 1
        payment = daily_b * num_days
        
        payment_date = pd.Timestamp(third_thu)
        if payment_date in df.index:
            df.loc[payment_date, 'company_b'] = payment

print(f"Company B total 2018: ${df.loc['2018', 'company_b'].sum():.2f}m")

In [ ]:
# ============================================================
# COMPANY C
# Daily charge of $0.18m per day.
# Weekdays: paid same day
# Weekends: accrues and paid on Monday
# ============================================================
df['company_c'] = 0.0
daily_c = 0.18  # $m

for idx in df.index:
    wd = idx.weekday()
    if wd < 5:  # weekday
        if wd == 0:  # Monday - gets Saturday and Sunday accruals too
            df.loc[idx, 'company_c'] = daily_c * 3  # Sat + Sun + Mon
        else:
            df.loc[idx, 'company_c'] = daily_c

print(f"Company C total 2018: ${df.loc['2018', 'company_c'].sum():.2f}m")

In [ ]:
# ============================================================
# COMPANY D
# Daily charge of $0.23m. Paid once per month on the 15th.
# Monthly payment = 0.23 * days_in_month
# Paid even on weekends
# ============================================================
df['company_d'] = 0.0
daily_d = 0.23  # $m

for year in range(2017, 2020):
    for month in range(1, 13):
        days_in_month = calendar.monthrange(year, month)[1]
        payment = daily_d * days_in_month
        payment_date = pd.Timestamp(date(year, month, 15))
        if payment_date in df.index:
            df.loc[payment_date, 'company_d'] = payment

print(f"Company D total 2018: ${df.loc['2018', 'company_d'].sum():.2f}m")

In [ ]:
# ============================================================
# COMPANY E
# Monthly charge = number_of_tuesdays * $1.96m
# Paid evenly across all weekdays of that month
# ============================================================
df['company_e'] = 0.0
rate_e = 1.96  # $m per Tuesday

for year in range(2017, 2020):
    for month in range(1, 13):
        # Count Tuesdays in the month
        month_start = pd.Timestamp(date(year, month, 1))
        days_in_month = calendar.monthrange(year, month)[1]
        month_end = pd.Timestamp(date(year, month, days_in_month))
        
        month_dates = df.loc[month_start:month_end]
        num_tuesdays = (month_dates['weekday'] == 1).sum()  # 1=Tuesday
        num_weekdays = month_dates['is_weekday'].sum()
        
        if num_weekdays > 0:
            total_monthly = rate_e * num_tuesdays
            daily_payment = total_monthly / num_weekdays
            
            weekday_mask = month_dates['is_weekday']
            for idx in month_dates[weekday_mask].index:
                df.loc[idx, 'company_e'] = daily_payment

print(f"Company E total 2018: ${df.loc['2018', 'company_e'].sum():.2f}m")

In [ ]:
# ============================================================
# COMPANY F
# 3 payments of $5.34m on 7th, 14th, 20th
# If weekend, paid on the Friday before
# ============================================================
df['company_f'] = 0.0
payment_f = 5.34  # $m
payment_days_f = [7, 14, 20]

def adjust_to_friday_before(d):
    """If date falls on weekend, move to preceding Friday."""
    wd = d.weekday()
    if wd == 5:  # Saturday -> Friday
        return d - timedelta(days=1)
    elif wd == 6:  # Sunday -> Friday
        return d - timedelta(days=2)
    return d

for year in range(2017, 2020):
    for month in range(1, 13):
        for pay_day in payment_days_f:
            d = date(year, month, pay_day)
            d = adjust_to_friday_before(d)
            ts = pd.Timestamp(d)
            if ts in df.index:
                df.loc[ts, 'company_f'] += payment_f

print(f"Company F total 2018: ${df.loc['2018', 'company_f'].sum():.2f}m")

In [ ]:
# ============================================================
# COMPANY G
# Single monthly payment of $9.23m
# Even months (Feb, Apr, Jun, Aug, Oct, Dec): 1st weekday
# Odd months (Jan, Mar, May, Jul, Sep, Nov): 3rd weekday
# ============================================================
df['company_g'] = 0.0
payment_g = 9.23  # $m

def nth_weekday_in_month(year, month, n):
    """Find the nth weekday (Mon-Fri) of the month. n is 1-based."""
    count = 0
    days_in_month = calendar.monthrange(year, month)[1]
    for day_num in range(1, days_in_month + 1):
        d = date(year, month, day_num)
        if d.weekday() < 5:  # weekday
            count += 1
            if count == n:
                return d
    return None

for year in range(2017, 2020):
    for month in range(1, 13):
        if month % 2 == 0:  # even month
            pay_date = nth_weekday_in_month(year, month, 1)
        else:  # odd month
            pay_date = nth_weekday_in_month(year, month, 3)
        
        if pay_date:
            ts = pd.Timestamp(pay_date)
            if ts in df.index:
                df.loc[ts, 'company_g'] = payment_g

print(f"Company G total 2018: ${df.loc['2018', 'company_g'].sum():.2f}m")

In [ ]:
# ============================================================
# COMPANY H
# $1.79m every Wednesday
# If payment falls on first 10 days of month: paid 1 day earlier
# If payment falls on last 10 days of month: paid 1 day later
# Otherwise (days 11 to last-10): paid on Wednesday as normal
# ============================================================
df['company_h'] = 0.0
payment_h = 1.79  # $m

for idx in df.index:
    if idx.weekday() == 2:  # Wednesday
        d = idx.date()
        days_in_month = calendar.monthrange(d.year, d.month)[1]
        
        if d.day <= 10:  # first 10 days
            adjusted = idx - pd.Timedelta(days=1)  # 1 day earlier (Tuesday)
        elif d.day > days_in_month - 10:  # last 10 days
            adjusted = idx + pd.Timedelta(days=1)  # 1 day later (Thursday)
        else:
            adjusted = idx  # stays on Wednesday
        
        if adjusted in df.index:
            df.loc[adjusted, 'company_h'] += payment_h

print(f"Company H total 2018: ${df.loc['2018', 'company_h'].sum():.2f}m")

In [ ]:
# ============================================================
# TOTAL DAILY CASH FLOW
# ============================================================
companies = ['company_a', 'company_b', 'company_c', 'company_d', 
             'company_e', 'company_f', 'company_g', 'company_h']
df['total'] = df[companies].sum(axis=1)

print(f"Total cash received in 2018: ${df.loc['2018', 'total'].sum():.2f}m")

In [ ]:
# ============================================================
# QUESTION 1: Cash from Company A between 7 Mar 2018 and 18 Oct 2018 (inclusive)
# Expected: D ($131.83m)
# ============================================================
q1_val = df.loc['2018-03-07':'2018-10-18', 'company_a'].sum()
print(f"Q1: Company A, 7 Mar - 18 Oct 2018: ${q1_val:.2f}m")

# Map to multiple choice
q1_options = {'A': 131.80, 'B': 131.81, 'C': 131.82, 'D': 131.83, 
              'E': 131.84, 'F': 131.85, 'G': 131.86, 'H': 131.87, 'I': 131.88}
q1_answer = min(q1_options.keys(), key=lambda k: abs(q1_options[k] - q1_val))
print(f"Q1 answer: {q1_answer}")

In [ ]:
# ============================================================
# QUESTION 2: Cash from Company B between months Apr 2018 and Oct 2018 (inclusive)
# "between the months" means Apr-Oct payments
# Expected: I ($75.95m)
# ============================================================
# Payments in Apr-Oct 2018 (3rd Thursdays)
q2_val = df.loc['2018-04-01':'2018-10-31', 'company_b'].sum()
print(f"Q2: Company B, Apr-Oct 2018: ${q2_val:.2f}m")

q2_options = {'A': 73.15, 'B': 73.50, 'C': 73.85, 'D': 74.20, 'E': 74.55, 
              'F': 74.90, 'G': 75.25, 'H': 75.60, 'I': 75.95}
q2_answer = min(q2_options.keys(), key=lambda k: abs(q2_options[k] - q2_val))
print(f"Q2 answer: {q2_answer}")

In [ ]:
# ============================================================
# QUESTION 3: Cash from Company C between 15 Jul 2018 and 24 Nov 2018 (inclusive)
# Expected: F ($23.94m)
# ============================================================
q3_val = df.loc['2018-07-15':'2018-11-24', 'company_c'].sum()
print(f"Q3: Company C, 15 Jul - 24 Nov 2018: ${q3_val:.2f}m")

q3_options = {'A': 23.04, 'B': 23.22, 'C': 23.40, 'D': 23.58, 'E': 23.76,
              'F': 23.94, 'G': 24.12, 'H': 24.30, 'I': 24.48}
q3_answer = min(q3_options.keys(), key=lambda k: abs(q3_options[k] - q3_val))
print(f"Q3 answer: {q3_answer}")

In [ ]:
# ============================================================
# QUESTION 4: Cash from Company D between months Apr 2018 and Oct 2018 (inclusive)
# Expected: E ($49.22m)
# ============================================================
q4_val = df.loc['2018-04-01':'2018-10-31', 'company_d'].sum()
print(f"Q4: Company D, Apr-Oct 2018: ${q4_val:.2f}m")

q4_options = {'A': 48.30, 'B': 48.53, 'C': 48.76, 'D': 48.99, 'E': 49.22,
              'F': 49.45, 'G': 49.68, 'H': 49.91, 'I': 50.14}
q4_answer = min(q4_options.keys(), key=lambda k: abs(q4_options[k] - q4_val))
print(f"Q4 answer: {q4_answer}")

In [ ]:
# ============================================================
# QUESTION 5: Cash from Company E between 2 Feb 2018 and 20 Aug 2018 (inclusive)
# Expected: H ($55.34m)
# ============================================================
q5_val = df.loc['2018-02-02':'2018-08-20', 'company_e'].sum()
print(f"Q5: Company E, 2 Feb - 20 Aug 2018: ${q5_val:.2f}m")

q5_options = {'A': 55.27, 'B': 55.28, 'C': 55.29, 'D': 55.30, 'E': 55.31,
              'F': 55.32, 'G': 55.33, 'H': 55.34, 'I': 55.35}
q5_answer = min(q5_options.keys(), key=lambda k: abs(q5_options[k] - q5_val))
print(f"Q5 answer: {q5_answer}")

In [ ]:
# ============================================================
# QUESTION 6: Cash from Company F on all Fridays in 2018
# Expected: E ($90.78m)
# ============================================================
mask_2018 = (df.index >= '2018-01-01') & (df.index <= '2018-12-31')
mask_friday = df['weekday'] == 4
q6_val = df.loc[mask_2018 & mask_friday, 'company_f'].sum()
print(f"Q6: Company F on Fridays 2018: ${q6_val:.2f}m")

q6_options = {'A': 69.42, 'B': 74.76, 'C': 80.10, 'D': 85.44, 'E': 90.78,
              'F': 96.12, 'G': 101.46, 'H': 106.80, 'I': 112.14}
q6_answer = min(q6_options.keys(), key=lambda k: abs(q6_options[k] - q6_val))
print(f"Q6 answer: {q6_answer}")

In [ ]:
# ============================================================
# QUESTION 7: How many times was cash received from Company G on the 3rd day of month in 2018
# Expected: D (3)
# ============================================================
mask_day3 = df['day'] == 3
q7_count = ((df.loc[mask_2018 & mask_day3, 'company_g']) > 0).sum()
print(f"Q7: Company G payments on 3rd of month in 2018: {q7_count}")

q7_options = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8}
q7_answer = min(q7_options.keys(), key=lambda k: abs(q7_options[k] - q7_count))
print(f"Q7 answer: {q7_answer}")

In [ ]:
# ============================================================
# QUESTION 8: Cash from Company H on all Tuesdays in 2018
# Expected: H ($28.64m)
# ============================================================
mask_tuesday = df['weekday'] == 1
q8_val = df.loc[mask_2018 & mask_tuesday, 'company_h'].sum()
print(f"Q8: Company H on Tuesdays 2018: ${q8_val:.2f}m")

q8_options = {'A': 16.11, 'B': 17.90, 'C': 19.69, 'D': 21.48, 'E': 23.27,
              'F': 25.06, 'G': 26.85, 'H': 28.64, 'I': 30.43}
q8_answer = min(q8_options.keys(), key=lambda k: abs(q8_options[k] - q8_val))
print(f"Q8 answer: {q8_answer}")

In [ ]:
# ============================================================
# QUESTION 9: Total cash from all companies between 1 Sep and 10 Sep 2018 (inclusive)
# Expected: A ($26.05m)
# ============================================================
q9_val = df.loc['2018-09-01':'2018-09-10', 'total'].sum()
print(f"Q9: Total all companies, 1-10 Sep 2018: ${q9_val:.2f}m")

q9_options = {'A': 26.05, 'B': 26.06, 'C': 26.07, 'D': 26.08, 'E': 26.09,
              'F': 26.10, 'G': 26.11, 'H': 26.12, 'I': 26.13}
q9_answer = min(q9_options.keys(), key=lambda k: abs(q9_options[k] - q9_val))
print(f"Q9 answer: {q9_answer}")

In [ ]:
# ============================================================
# QUESTION 10: Total cash from all companies in 2018
# Expected: $983.01m
# ============================================================
q10_val = df.loc['2018', 'total'].sum()
print(f"Q10: Total all companies 2018: ${q10_val:.2f}m")
q10_answer = f"${q10_val:.2f}m"

In [ ]:
# ============================================================
# QUESTION 11: Company F payments now on 6th, 13th, 19th (still adjusted to before weekend)
# Cash from Company F on all Fridays in 2018
# Expected: C ($74.76m)
# ============================================================
# Recalculate Company F with new payment days
df['company_f_q11'] = 0.0
payment_days_f_q11 = [6, 13, 19]

for year in range(2017, 2020):
    for month in range(1, 13):
        for pay_day in payment_days_f_q11:
            d = date(year, month, pay_day)
            d = adjust_to_friday_before(d)
            ts = pd.Timestamp(d)
            if ts in df.index:
                df.loc[ts, 'company_f_q11'] += payment_f

q11_val = df.loc[mask_2018 & mask_friday, 'company_f_q11'].sum()
print(f"Q11: Company F (new dates) on Fridays 2018: ${q11_val:.2f}m")

q11_options = {'A': 64.08, 'B': 69.42, 'C': 74.76, 'D': 80.10, 'E': 85.44,
               'F': 90.78, 'G': 96.12, 'H': 101.46, 'I': 106.80}
q11_answer = min(q11_options.keys(), key=lambda k: abs(q11_options[k] - q11_val))
print(f"Q11 answer: {q11_answer}")

In [ ]:
# ============================================================
# QUESTION 12: Company H now pays on Tuesdays instead of Wednesdays
# (same first/last 10 day adjustments)
# Total from all companies between 1 Jun and 31 Dec 2018 (inclusive)
# Note: Q11 says "reset your values" so Company F is back to original (7th, 14th, 20th)
# Expected: $575.83m
# ============================================================
df['company_h_q12'] = 0.0

for idx in df.index:
    if idx.weekday() == 1:  # Tuesday instead of Wednesday
        d = idx.date()
        days_in_month = calendar.monthrange(d.year, d.month)[1]
        
        if d.day <= 10:  # first 10 days
            adjusted = idx - pd.Timedelta(days=1)  # 1 day earlier (Monday)
        elif d.day > days_in_month - 10:  # last 10 days
            adjusted = idx + pd.Timedelta(days=1)  # 1 day later (Wednesday)
        else:
            adjusted = idx  # stays on Tuesday
        
        if adjusted in df.index:
            df.loc[adjusted, 'company_h_q12'] += payment_h

# Total for Q12: all companies with original F but modified H
q12_total = (df.loc['2018-06-01':'2018-12-31', 'company_a'].sum() +
             df.loc['2018-06-01':'2018-12-31', 'company_b'].sum() +
             df.loc['2018-06-01':'2018-12-31', 'company_c'].sum() +
             df.loc['2018-06-01':'2018-12-31', 'company_d'].sum() +
             df.loc['2018-06-01':'2018-12-31', 'company_e'].sum() +
             df.loc['2018-06-01':'2018-12-31', 'company_f'].sum() +
             df.loc['2018-06-01':'2018-12-31', 'company_g'].sum() +
             df.loc['2018-06-01':'2018-12-31', 'company_h_q12'].sum())
print(f"Q12: Total all companies (H modified), Jun-Dec 2018: ${q12_total:.2f}m")
q12_answer = f"${q12_total:.2f}m"

In [ ]:
# ============================================================
# FINAL ANSWERS
# ============================================================
answers = {
    "question1": q1_answer,
    "question2": q2_answer,
    "question3": q3_answer,
    "question4": q4_answer,
    "question5": q5_answer,
    "question6": q6_answer,
    "question7": q7_answer,
    "question8": q8_answer,
    "question9": q9_answer,
    "question10": q10_answer,
    "question11": q11_answer,
    "question12": q12_answer,
}

print("\n=== FINAL ANSWERS ===")
for k, v in answers.items():
    print(f"  {k}: {v}")